# Step 2 — Run Nanbeige 4.3 33B on a Kaggle GPU and expose an OpenAI-compatible API

This notebook mounts your reusable **Nanbeige GGUF Dataset** and prebuilt CUDA-enabled **`llama-cpp-bin` Dataset**, starts `llama-server`, and opens a Cloudflare Quick Tunnel.

## Architecture

```text
Kaggle GPU session
  Nanbeige_Nanbeige4.3-33B-Q4_K_M.gguf
          ↕
  llama-server :8080  ← OpenAI-compatible /v1 API
          ↕
  cloudflared Quick Tunnel
          ↕
  https://<temporary-name>.trycloudflare.com
          ↕
  VS Code / Cline / other OpenAI-compatible client
```

## Before running

1. Set the notebook accelerator to **GPU T4 x2** and enable Internet.
2. Attach your private **`nanbeige-4-3-33b-gguf`** Dataset.
3. Attach your **`llama-cpp-bin`** Dataset. It must be a recent CUDA-enabled `llama.cpp` build that recognizes the Nanbeige architecture.
4. In **Add-ons → Secrets**, optionally add `NANBEIGE_API_KEY` and enable it for this notebook. If you don't, the notebook generates a random key for this session and prints it once.
5. If your dataset slug or GGUF filename differs, edit Cell 1.

**Context warning:** this is configured to *attempt* 262,144 tokens using Q4 KV cache. That is an ambitious target on 2×T4: model weights, KV cache, and compute buffers all consume VRAM. If it fails to load or runs out of memory, lower `CTX_SIZE` to `131072`, `65536`, or `32768`. A model's advertised context length does not guarantee that it fits in your available VRAM.

The model is text-only. Do not pass `--mmproj`.

## Cell 1 — Configuration and dataset discovery

In [ ]:
import os
import json
import shutil
import subprocess
import secrets
from pathlib import Path

# Change these only if your attached Kaggle dataset/filename is different.
KAGGLE_USERNAME = ""  # Optional: your Kaggle username; can also be an environment variable.
MODEL_DATASET = "nanbeige-4-3-33b-gguf"
MODEL_FILE = "Nanbeige_Nanbeige4.3-33B-Q4_K_M.gguf"
LLAMA_DATASET = "llama-cpp-bin"

PORT = 8080
CTX_SIZE = 262144       # Target context. Reduce if model loading fails or VRAM runs out.
N_GPU_LAYERS = 999
PARALLEL = 1
SPLIT_MODE = "layer"

# Quantized KV cache can substantially reduce memory use at long context lengths.
# If this llama.cpp build/model rejects q4_0 KV cache, try "q8_0" or "f16".
CACHE_TYPE_K = "q4_0"
CACHE_TYPE_V = "q4_0"

TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")

# Prefer a Kaggle Secret named NANBEIGE_API_KEY. Otherwise generate a session-only key.
API_KEY = get_secret("NANBEIGE_API_KEY")
if not API_KEY:
    API_KEY = secrets.token_urlsafe(32)
    print("No NANBEIGE_API_KEY secret found; generated a random key for this session.")
    print("Save this key somewhere private; it will not persist when the Kaggle session ends.")
    print("API key:", API_KEY)
else:
    print("Loaded API key from Kaggle Secrets.")

def find_dataset(slug):
    candidates = []
    if KAGGLE_USERNAME:
        candidates.append(Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{slug}"))
    candidates.append(Path(f"/kaggle/input/{slug}"))
    for p in candidates:
        if p.exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        matches = [p for p in root.rglob(slug) if p.is_dir() and p.name == slug]
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Dataset '{slug}' not found. Attach it in the notebook's Input panel. "
        f"Checked: {', '.join(map(str, candidates))}"
    )

MODEL_DIR = find_dataset(MODEL_DATASET)
LLAMA_BIN_DIR = find_dataset(LLAMA_DATASET)
MODEL_PATH = MODEL_DIR / MODEL_FILE
WORKING = Path("/kaggle/working")
LLAMA_SERVER = WORKING / "llama-server"

if not MODEL_PATH.exists():
    ggufs = list(MODEL_DIR.rglob("*.gguf"))
    raise FileNotFoundError(
        f"Expected model file not found: {MODEL_PATH}\n"
        f"GGUF files found: {[p.name for p in ggufs]}"
    )
if not (LLAMA_BIN_DIR / "llama-server").exists():
    raise FileNotFoundError(f"llama-server not found in {LLAMA_BIN_DIR}")

print("Configuration validated.")
print("Model:", MODEL_PATH)
print("Model size: %.2f GiB" % (MODEL_PATH.stat().st_size / (1024**3)))
print("Binaries:", LLAMA_BIN_DIR)
print("Requested context:", CTX_SIZE)
print("KV cache types:", CACHE_TYPE_K, CACHE_TYPE_V)

## Cell 2 — Check available GPU memory

In [ ]:
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError("No NVIDIA GPU detected. Set the notebook accelerator to GPU T4 x2.")

total_mib = 0
print(f"{'GPU':<5} {'Name':<24} {'Total VRAM':>12} {'Free VRAM':>12}")
print("-" * 59)
for i, line in enumerate(result.stdout.strip().splitlines()):
    parts = [x.strip() for x in line.split(",")]
    name, total, free = parts
    total_mb = int(total.replace(" MiB", ""))
    free_mb = int(free.replace(" MiB", ""))
    total_mib += total_mb
    print(f"{i:<5} {name:<24} {total_mb/1024:>9.1f} GiB {free_mb/1024:>9.1f} GiB")
print("-" * 59)
print(f"Total VRAM across visible GPUs: {total_mib/1024:.1f} GiB")
print("Remember: VRAM is split across GPUs; total VRAM is not one contiguous pool.")

## Cell 3 — Copy `llama.cpp` binaries and libraries into writable storage

In [ ]:
WORKING.mkdir(parents=True, exist_ok=True)

for name in ["llama-server", "llama-cli", "llama-mtmd-cli"]:
    src = LLAMA_BIN_DIR / name
    if src.exists():
        dst = WORKING / name
        shutil.copy2(src, dst)
        dst.chmod(0o755)

# Copy shared libraries from the dataset, including libraries in subdirectories.
so_files = list(LLAMA_BIN_DIR.rglob("*.so*"))
copied = 0
for src in so_files:
    dst = WORKING / src.name
    if src.resolve() == dst.resolve():
        continue
    shutil.copy2(src, dst)
    copied += 1

os.environ["LD_LIBRARY_PATH"] = f"{WORKING}:{os.environ.get('LD_LIBRARY_PATH', '')}"
env = {**os.environ, "LD_LIBRARY_PATH": os.environ["LD_LIBRARY_PATH"]}

test = subprocess.run(
    [str(LLAMA_SERVER), "--version"],
    capture_output=True, text=True, env=env
)
if test.returncode != 0:
    print(test.stdout)
    print(test.stderr[-5000:])
    raise RuntimeError("llama-server failed --version. Check the binary dataset and CUDA dependencies.")

version_file = LLAMA_BIN_DIR / "version.json"
if version_file.exists():
    info = json.loads(version_file.read_text())
    print("llama.cpp binary check passed.")
    print("Commit:", info.get("llama_cpp_commit", "unknown"))
    print("Commit date:", info.get("commit_date", "unknown"))
    print("Version:", info.get("llama_server_ver", (test.stdout or test.stderr).strip().splitlines()[0]))
    print("CUDA target:", info.get("cuda_arch", "unknown"))
else:
    print("llama.cpp binary check passed:", (test.stdout or test.stderr).strip().splitlines()[0])
print(f"Copied {copied} shared libraries.")

## Cell 4 — Verify server options

In [ ]:
env = {**os.environ, "LD_LIBRARY_PATH": f"{WORKING}:{os.environ.get('LD_LIBRARY_PATH', '')}"}
help_result = subprocess.run(
    [str(LLAMA_SERVER), "--help"],
    capture_output=True, text=True, env=env, timeout=30
)
help_text = (help_result.stdout or "") + "\n" + (help_result.stderr or "")
if help_result.returncode != 0:
    print(help_text[-4000:])
    raise RuntimeError("llama-server --help failed.")

required_flags = ["--ctx-size", "--parallel", "--api-key"]
missing = [flag for flag in required_flags if flag not in help_text]
if missing:
    print(help_text[-4000:])
    raise RuntimeError(f"This llama.cpp build is missing expected options: {missing}")

for flag in ["--cache-type-k", "--cache-type-v", "--split-mode"]:
    if flag not in help_text:
        print(f"WARNING: {flag} was not found in --help; this build may be too old.")

print("Server CLI runs. Nanbeige architecture compatibility is conclusively tested during model loading.")
print("If logs report an unknown architecture, rebuild llama.cpp from a revision that supports this model.")

## Cell 5 — Install Cloudflare Tunnel

In [ ]:
cloudflared = Path("/usr/local/bin/cloudflared")
download = subprocess.run(
    ["bash", "-lc",
     "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
     "-O /tmp/cloudflared && chmod +x /tmp/cloudflared && install /tmp/cloudflared /usr/local/bin/cloudflared"],
    capture_output=True, text=True
)
if download.returncode != 0:
    print(download.stdout)
    print(download.stderr[-3000:])
    raise RuntimeError("Could not install cloudflared.")

version = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)
print((version.stdout or version.stderr).strip())

## Cell 6 — Launch Nanbeige server

In [ ]:
import time
import requests
import signal

# Stop a prior server if this cell is rerun.
subprocess.run(["pkill", "-x", "llama-server"], capture_output=True)
time.sleep(1)

env = {**os.environ, "LD_LIBRARY_PATH": f"{WORKING}:{os.environ.get('LD_LIBRARY_PATH', '')}"}

cmd = [
    str(LLAMA_SERVER),
    "--model", str(MODEL_PATH),
    "--alias", "nanbeige-4.3-33b",
    "--host", "127.0.0.1",  # Tunnel connects locally; don't expose port directly on the notebook network.
    "--port", str(PORT),
    "--n-gpu-layers", str(N_GPU_LAYERS),
    "--ctx-size", str(CTX_SIZE),
    "--parallel", str(PARALLEL),
    "--split-mode", SPLIT_MODE,
    "--cache-type-k", CACHE_TYPE_K,
    "--cache-type-v", CACHE_TYPE_V,
    "--temp", str(TEMPERATURE),
    "--top-p", str(TOP_P),
    "--top-k", str(TOP_K),
    "--api-key", API_KEY,
    "--log-file", "/tmp/llama-server.log",
]
# Enable flash attention when this build supports the flag.
if "--flash-attn" in help_text:
    cmd += ["--flash-attn", "on"]

print("Starting Nanbeige 4.3 33B...")
print("Requested context:", CTX_SIZE)
print("KV cache:", CACHE_TYPE_K, CACHE_TYPE_V)
print("GPU layers:", N_GPU_LAYERS)
print("Parallel slots:", PARALLEL)

server_proc = subprocess.Popen(
    cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env
)

timeout_seconds = 900
interval = 10
elapsed = 0
server_ok = False

while elapsed < timeout_seconds:
    if server_proc.poll() is not None:
        break
    try:
        response = requests.get(
            f"http://127.0.0.1:{PORT}/health",
            headers={"Authorization": f"Bearer {API_KEY}"},
            timeout=3
        )
        if response.status_code == 200 and response.json().get("status") == "ok":
            server_ok = True
            break
    except Exception:
        pass
    print(f"[{elapsed:>3}s] Model is loading...", flush=True)
    time.sleep(interval)
    elapsed += interval

if not server_ok:
    print("\nLast server log lines:")
    try:
        print("\n".join(Path("/tmp/llama-server.log").read_text(errors="replace").splitlines()[-80:]))
    except Exception as e:
        print("Could not read log:", e)
    if server_proc.poll() is not None:
        print("Server exited with code:", server_proc.returncode)
    raise RuntimeError(
        "Nanbeige did not become healthy. If this is an OOM, reduce CTX_SIZE "
        "to 131072, 65536, or 32768 and rerun from Cell 6."
    )

print(f"\nNanbeige is ready after approximately {elapsed} seconds.")
print(f"Local OpenAI-compatible endpoint: http://127.0.0.1:{PORT}/v1")

## Cell 7 — Start Cloudflare Quick Tunnel

In [ ]:
import re

tunnel_log_path = Path("/tmp/cloudflare.log")
tunnel_log_path.write_text("")

tunnel_log_handle = open(tunnel_log_path, "a")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log_handle,
    stderr=tunnel_log_handle,
)

tunnel_url = None
for _ in range(45):
    time.sleep(2)
    log_text = tunnel_log_path.read_text(errors="replace")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    if tunnel_proc.poll() is not None:
        break

if not tunnel_url:
    print(tunnel_log_path.read_text(errors="replace")[-4000:])
    raise RuntimeError("Could not obtain a Cloudflare Quick Tunnel URL.")

print("=" * 68)
print("NANBEIGE SERVER URL")
print(tunnel_url)
print("=" * 68)
print("OpenAI-compatible base URL:", tunnel_url + "/v1")
print("Model name:", "nanbeige-4.3-33b")
print("Authentication: enabled")
print("Use the API key printed in Cell 1 (or stored in your Kaggle Secret).")
print("The URL is temporary and will change when a new tunnel/session is created.")

## Cell 8 — API smoke test

In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}"}

try:
    r = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=headers, timeout=15)
    print("HTTP status:", r.status_code)
    print(r.text[:2000])
    r.raise_for_status()

    # Tiny chat-completions test to verify that generation works too.
    payload = {
        "model": "nanbeige-4.3-33b",
        "messages": [{"role": "user", "content": "Reply with exactly: API works."}],
        "max_tokens": 16,
        "temperature": 0
    }
    chat = requests.post(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        headers={**headers, "Content-Type": "application/json"},
        json=payload,
        timeout=180
    )
    print("Chat test HTTP status:", chat.status_code)
    print(chat.text[:3000])
    chat.raise_for_status()
except Exception as e:
    print("API smoke test failed:", repr(e))
    print("Inspect /tmp/llama-server.log")

## Cell 9 — Keep the server alive and report health

In [ ]:
import datetime

start_time = datetime.datetime.now()
check_interval = 300

print("Nanbeige server started:", start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("Public URL:", tunnel_url)
print("Base API URL:", tunnel_url + "/v1")
print("Model alias:", "nanbeige-4.3-33b")
print("Stop by interrupting this cell.")
print("-" * 68)

try:
    while True:
        time.sleep(check_interval)
        elapsed_td = datetime.datetime.now() - start_time
        h, rem = divmod(int(elapsed_td.total_seconds()), 3600)
        m = rem // 60
        try:
            r = requests.get(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": f"Bearer {API_KEY}"},
                timeout=5
            )
            status = "healthy" if r.status_code == 200 and r.json().get("status") == "ok" else f"HTTP {r.status_code}"
        except Exception:
            status = "unreachable"
        print(f"[{datetime.datetime.now():%H:%M:%S}] Uptime {h}h {m:02d}m | {status}", flush=True)
        if status == "unreachable":
            print("Server appears unreachable. Check /tmp/llama-server.log")
            break
except KeyboardInterrupt:
    print("\nStopping server and tunnel...")
    for proc in [server_proc, tunnel_proc]:
        try:
            proc.terminate()
        except Exception:
            pass
    try:
        tunnel_log_handle.close()
    except Exception:
        pass
    print("Stopped.")

## VS Code / Cline settings

- **Provider:** OpenAI-compatible / OpenAI
- **Base URL:** `https://<your-tunnel>.trycloudflare.com/v1`
- **API key:** the key printed in Cell 1, or the value of your `NANBEIGE_API_KEY` Kaggle Secret
- **Model ID:** `nanbeige-4.3-33b`

## Troubleshooting

- **Model file not found:** check the attached dataset slug and the exact `.gguf` filename in Cell 1.
- **Unknown architecture / failed model load:** rebuild `llama-cpp-bin` from a `llama.cpp` revision that supports Nanbeige 4.3.
- **Out of VRAM:** lower `CTX_SIZE` first. Try `131072`, `65536`, then `32768`. If the quantized KV cache is unsupported, switch `CACHE_TYPE_K` and `CACHE_TYPE_V` to `q8_0` or `f16` (these use more memory).
- **KV cache type unsupported:** inspect `/tmp/llama-server.log`; switch cache types as above.
- **Tunnel works but client cannot connect:** check the `/v1` suffix, model ID, and Bearer API key.
- **Security:** the tunnel is publicly reachable while active, but requests require the generated/configured API key. Never share the key. Stop the notebook when finished.

**Important:** 262K is a requested context setting, not a guarantee that all 262K tokens will be usable on 2×T4. Confirm actual context and generation with your real coding-agent prompt before relying on it.